In [1]:
import pandas as pd
from scipy.stats import spearmanr, kendalltau
import numpy as np
from tqdm.auto import tqdm
from copy import deepcopy

from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [2]:
root_path = "/home/stefan/ioai-prep/kits/neoai/codestars"
seed = 42

# Data

In [3]:
train = pd.read_parquet(f'{root_path}/train.parquet')
train = train[["contestId", "handle", "rank"]]

train_results = pd.read_parquet(f'{root_path}/train_results.parquet')
problems = pd.read_parquet(f'{root_path}/problems.parquet')
contest = pd.read_parquet(f'{root_path}/contest.parquet')

test = pd.read_parquet(f'{root_path}/test.parquet')

train.head()

,contestId,handle,rank
0,566,106998,1
1,566,363813,2
2,566,163675,3
3,566,3987,4
4,566,364998,5


In [15]:
problems[problems["contestId"] == test.iloc[0]["contestId"]]

,contestId,index,points,rating,tags
99,568,A,500.0,800.0,"greedy,implementation,math"
100,568,B,1000.0,1000.0,"bitmasks,greedy,strings"
101,568,C,1500.0,1300.0,"greedy,sortings"
102,568,D,2000.0,1800.0,"data structures,dp,graphs,sortings"
103,568,E,2500.0,2400.0,"data structures,dfs and similar,dp,dsu,greedy,..."
104,568,F,2750.0,2400.0,"bitmasks,combinatorics,dp,math,number theory"


In [16]:
contest[contest["contestId"] == test.iloc[0]["contestId"]]

,contestId,division,type,durationSeconds,startTime
14,568,Div2,CF,7200,2025-10-02 14:35:00


In [ ]:
handle_means = train.groupby("handle")["rank"].median()
global_mean = handle_means.median()

def get_contest_features():
    target_indices = ["A", "B", "C", "D", "E", "F"]
    subset_probs = problems[problems["index"].isin(target_indices)].copy()

    prob_pivot = subset_probs.pivot(
        index="contestId", columns="index", values=["points", "rating"]
    )
    prob_pivot.columns = [f"{idx}_{val}" for val, idx in prob_pivot.columns]

    contest_tags = (
        problems.groupby("contestId")["tags"]
        .apply(lambda x: ",".join(set(",".join(x.dropna()).split(","))))
        .rename("all_tags")
    )

    meta = contest[["contestId", "division", "type", "durationSeconds"]].copy()
    meta = meta.merge(prob_pivot, on="contestId", how="left")
    meta = meta.merge(contest_tags, on="contestId", how="left")
    return meta


contest_meta = get_contest_features()


def prep_df(df: pd.DataFrame):
    df["mean_rank"] = df["handle"].map(handle_means).fillna(global_mean)

    df = df.merge(contest_meta, on="contestId", how="left")

    df["all_tags"] = df["all_tags"].fillna("none")

    relevant_tags = ["dp", "combinatorics", "dfs", "graphs", "bitmasks", "implementation", "math", "strings", "greedy", "number theory", "data structures", "dsu", "sortings"]
    for tag in relevant_tags:
        df[f"tag_{tag}"] = df["all_tags"].str.contains(tag)
    df = df.drop(["all_tags"], axis=1)

    return df


df_train = prep_df(train)
df_test = prep_df(test)

df_train.head()

,contestId,handle,rank,mean_rank,division,type,durationSeconds,A_points,B_points,C_points,...,tag_graphs,tag_bitmasks,tag_implementation,tag_math,tag_strings,tag_greedy,tag_number theory,tag_data structures,tag_dsu,tag_sortings
0,566,106998,1,108.0,Div1,CF,7200,500.0,1250.0,NaN,...,False,False,False,True,True,True,False,True,False,False
1,566,363813,2,187.0,Div1,CF,7200,500.0,1250.0,NaN,...,False,False,False,True,True,True,False,True,False,False
2,566,163675,3,191.0,Div1,CF,7200,500.0,1250.0,NaN,...,False,False,False,True,True,True,False,True,False,False
3,566,3987,4,3528.0,Div1,CF,7200,500.0,1250.0,NaN,...,False,False,False,True,True,True,False,True,False,False
4,566,364998,5,206.0,Div1,CF,7200,500.0,1250.0,NaN,...,False,False,False,True,True,True,False,True,False,False


In [7]:
def recall_at_k(y_true, y_pred, k=100):
    pred_topk = set(np.argsort(y_pred)[:k])
    true_topk = set(np.where(np.array(y_true) <= k)[0])
    if len(true_topk) == 0:
        return 0.0
    return len(pred_topk & true_topk) / len(true_topk)


def kendall_at_1000(y_true, y_pred):
    tau = kendalltau(y_true, y_pred).statistic
    return tau if not np.isnan(tau) else 0.0


def competition_metric(y_true, y_pred, k=100):
    return 0.3 * recall_at_k(y_true, y_pred, k) + 0.7 * kendall_at_1000(y_true, y_pred)

In [8]:
cat_features = ["division", "type"]
num_features = [
    col
    for col in df_train.columns
    if col not in cat_features + ["contestId", "handle", "rank"]
]

preprocessor = ColumnTransformer(transformers=[
    (
        "num",
        Pipeline([("imp", SimpleImputer(strategy="mean")), ("std", StandardScaler())]),
        num_features,
    ),
    (
        "cat",
        Pipeline([
            ("imp", SimpleImputer(strategy="constant", fill_value="missing")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]),
        cat_features,
    )]
)

X_train_proc = preprocessor.fit_transform(df_train)
X_test_proc = preprocessor.transform(df_test)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X_train_proc, df_train["rank"], test_size=0.2, random_state=seed)

In [10]:
def evaluate(model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return competition_metric(y_test, y_pred)

In [11]:
cb = CatBoostRegressor(iterations=400, depth=3, random_seed=seed)
evaluate(cb)

Learning rate set to 0.326826
0:	learn: 3464.1792680	total: 369ms	remaining: 2m 27s
1:	learn: 3086.0491501	total: 600ms	remaining: 1m 59s
2:	learn: 2875.1229081	total: 853ms	remaining: 1m 52s
3:	learn: 2760.1375659	total: 1.15s	remaining: 1m 54s
4:	learn: 2698.4525469	total: 1.41s	remaining: 1m 51s
5:	learn: 2660.7687577	total: 1.67s	remaining: 1m 49s
6:	learn: 2639.0189977	total: 1.95s	remaining: 1m 49s
7:	learn: 2625.4044566	total: 2.2s	remaining: 1m 47s
8:	learn: 2615.2035020	total: 2.46s	remaining: 1m 47s
9:	learn: 2608.4267773	total: 2.65s	remaining: 1m 43s
10:	learn: 2603.4109345	total: 2.89s	remaining: 1m 42s
11:	learn: 2600.0765723	total: 3.12s	remaining: 1m 40s
12:	learn: 2593.0747260	total: 3.34s	remaining: 1m 39s
13:	learn: 2588.6910215	total: 3.56s	remaining: 1m 38s
14:	learn: 2585.7894138	total: 3.8s	remaining: 1m 37s
15:	learn: 2581.8838204	total: 4.03s	remaining: 1m 36s
16:	learn: 2580.0201582	total: 4.21s	remaining: 1m 34s
17:	learn: 2575.7076991	total: 4.46s	remaining:

np.float64(0.43871983400452286)

# Solution

In [12]:
X_feats = df_test.drop(["contestId", "handle"], axis=1)

In [13]:
sub = deepcopy(test)

sub["rank"] = cb.predict(X_test_proc)
sub["Id"] = sub["contestId"].astype(str) + "_" + sub["handle"].astype(str)
sub = sub[["Id", "rank"]]

sub.head()

,Id,rank
0,568_309817,1861.077564
1,568_102551,3168.883744
2,568_205873,3870.745379
3,568_38199,1508.590335
4,568_528524,5858.743016


In [14]:
sub[["rank", "Id"]].to_csv(f"{root_path}/predict1.csv", index=None)